In [5]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [6]:
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [37]:
df = pd.read_csv('adult_income_cleaned.csv')
 
print("Dataset loaded")
print("Shape:", df.shape)
print()
print(df.head())

Dataset loaded
Shape: (32521, 18)

   age         workclass  fnlwgt  education  education_num  \
0   39         State-gov   77516  Bachelors             13   
1   50  Self-emp-not-inc   83311  Bachelors             13   
2   38           Private  215646    HS-grad              9   
3   53           Private  234721       11th              7   
4   28           Private  338409  Bachelors             13   

       marital_status         occupation   relationship   race     sex  \
0       Never-married       Adm-clerical  Not-in-family  White    Male   
1  Married-civ-spouse    Exec-managerial        Husband  White    Male   
2            Divorced  Handlers-cleaners  Not-in-family  White    Male   
3  Married-civ-spouse  Handlers-cleaners        Husband  Black    Male   
4  Married-civ-spouse     Prof-specialty           Wife  Black  Female   

   capital_gain  capital_loss  hours_per_week native_country income  \
0          2174             0              40  United-States  <=50K   
1    

In [8]:
print("\n========== BASIC STATISTICS ==========")
print(df.describe().round(2))


========== BASIC STATISTICS ==========
            age      fnlwgt  education_num  capital_gain  capital_loss  \
count  32521.00    32521.00       32521.00      32521.00      32521.00   
mean      38.57   189783.16          10.08       1078.97         87.41   
std       13.55   105566.31           2.57       7389.74        403.20   
min       17.00    12285.00           1.00          0.00          0.00   
25%       28.00   117816.00           9.00          0.00          0.00   
50%       37.00   178356.00          10.00          0.00          0.00   
75%       48.00   237044.00          12.00          0.00          0.00   
max       78.00  1484705.00          16.00      99999.00       4356.00   

       hours_per_week  capital_net  is_high_earner  
count        32521.00     32521.00        32521.00  
mean            41.07       991.56            0.24  
std              6.21      7413.46            0.43  
min             32.00     -4356.00            0.00  
25%             40.00       

In [9]:
print("\n========== CATEGORICAL COLUMN STATS ==========")
print(df.describe(include='object'))


========== CATEGORICAL COLUMN STATS ==========
       workclass education      marital_status      occupation relationship  \
count      32521     32521               32521           32521        32521   
unique         8        16                   7              14            6   
top      Private   HS-grad  Married-civ-spouse  Prof-specialty      Husband   
freq       24494     10494               14969            5972        13186   

         race    sex native_country income   age_group  
count   32521  32521          32521  32521       32521  
unique      5      2             41      2           4  
top     White   Male  United-States  <=50K  Middle Age  
freq    27780  21771          29720  24682       11168  


In [10]:

print("\n========== VALUE COUNTS ==========")
cat_cols = ['workclass', 'education', 'marital_status',
            'occupation', 'race', 'sex', 'income', 'age_group']
 
for col in cat_cols:
    print(f"\n{col}:")
    print(df[col].value_counts().to_string())


========== VALUE COUNTS ==========

workclass:
workclass
Private             24494
Self-emp-not-inc     2540
Local-gov            2093
State-gov            1297
Self-emp-inc         1116
Federal-gov           960
Without-pay            14
Never-worked            7

education:
education
HS-grad         10494
Some-college     7270
Bachelors        5351
Masters          1721
Assoc-voc        1382
11th             1174
Assoc-acdm       1067
10th              933
7th-8th           645
Prof-school       576
9th               514
12th              433
Doctorate         413
5th-6th           332
1st-4th           166
Preschool          50

marital_status:
marital_status
Married-civ-spouse       14969
Never-married            10652
Divorced                  4441
Separated                 1025
Widowed                    993
Married-spouse-absent      418
Married-AF-spouse           23

occupation:
occupation
Prof-specialty       5972
Craft-repair         4094
Exec-managerial      4065
Adm-cleri

In [11]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Income Distribution', fontsize=14, fontweight='bold')
 
income_counts = df['income'].value_counts()

In [12]:
bars = axes[0].bar(income_counts.index, income_counts.values,
                   color=['#2ecc71', '#e74c3c'],
                   edgecolor='black', width=0.4)
axes[0].set_title('Count of Each Income Class')
axes[0].set_xlabel('Income')
axes[0].set_ylabel('Number of People')
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 100,
                 f'{int(bar.get_height()):,}',
                 ha='center', fontweight='bold')

In [13]:
axes[1].pie(income_counts.values,
            labels=income_counts.index,
            autopct='%1.1f%%',
            colors=['#2ecc71', '#e74c3c'],
            startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Income Split Percentage')
 
plt.tight_layout()
plt.savefig('eda_01_income_distribution.png', dpi=150, bbox_inches='tight')
plt.close()
print("\nSaved: eda_01_income_distribution.png")
print(f"Insight: {income_counts['<=50K']:,} people ({income_counts['<=50K']/len(df)*100:.1f}%) earn <=50K")
print(f"         {income_counts['>50K']:,} people ({income_counts['>50K']/len(df)*100:.1f}%) earn >50K")
print("         Dataset is imbalanced — more low earners than high earners")


Saved: eda_01_income_distribution.png
Insight: 24,682 people (75.9%) earn <=50K
         7,839 people (24.1%) earn >50K
         Dataset is imbalanced — more low earners than high earners


In [14]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Age Analysis', fontsize=14, fontweight='bold')
 
# overall age distribution
axes[0].hist(df['age'], bins=30, color='steelblue',
             edgecolor='black', alpha=0.8)
axes[0].axvline(df['age'].mean(), color='red', linestyle='--',
                label=f"Mean = {df['age'].mean():.1f}")
axes[0].axvline(df['age'].median(), color='orange', linestyle='--',
                label=f"Median = {df['age'].median():.1f}")
axes[0].set_title('Age Distribution (Overall)')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')
axes[0].legend()

In [15]:
# age split by income
df[df['income'] == '<=50K']['age'].hist(bins=30, ax=axes[1],
    alpha=0.6, color='#2ecc71', edgecolor='black', label='<=50K')
df[df['income'] == '>50K']['age'].hist(bins=30, ax=axes[1],
    alpha=0.6, color='#e74c3c', edgecolor='black', label='>50K')
axes[1].set_title('Age Distribution by Income Group')
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Count')
axes[1].legend()
 
plt.tight_layout()
plt.savefig('eda_02_age_analysis.png', dpi=150, bbox_inches='tight')
plt.close()
print("\nSaved: eda_02_age_analysis.png")
print(f"Insight: Average age of <=50K earners: {df[df['income']=='<=50K']['age'].mean():.1f}")
print(f"         Average age of >50K earners : {df[df['income']=='>50K']['age'].mean():.1f}")
print("         High earners tend to be older on average")


Saved: eda_02_age_analysis.png
Insight: Average age of <=50K earners: 36.8
         Average age of >50K earners : 44.2
         High earners tend to be older on average


In [16]:
fig, axes = plt.subplots(2, 1, figsize=(14, 11))
fig.suptitle('Education Analysis', fontsize=14, fontweight='bold')

Text(0.5, 0.98, 'Education Analysis')

In [17]:
edu_counts = df['education'].value_counts()
axes[0].bar(edu_counts.index, edu_counts.values,
            color='cornflowerblue', edgecolor='black', alpha=0.85)
axes[0].set_title('Number of People per Education Level')
axes[0].set_xlabel('Education')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

In [18]:
edu_income = df.groupby('education')['is_high_earner'].mean().sort_values(ascending=False) * 100
colors = ['#c0392b' if x > 40 else '#f39c12' if x > 20 else '#27ae60'
          for x in edu_income.values]
bars = axes[1].bar(edu_income.index, edu_income.values,
                   color=colors, edgecolor='black', alpha=0.85)
axes[1].set_title('% of High Earners (>50K) by Education Level\n(Red=High, Orange=Medium, Green=Low)')
axes[1].set_xlabel('Education')
axes[1].set_ylabel('% Earning >50K')
axes[1].tick_params(axis='x', rotation=45)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0f}%'))
for bar in bars:
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.5,
                 f'{bar.get_height():.1f}%',
                 ha='center', fontsize=8, fontweight='bold')
 
plt.tight_layout()
plt.savefig('eda_03_education_analysis.png', dpi=150, bbox_inches='tight')
plt.close()
print("\nSaved: eda_03_education_analysis.png")
top_edu = edu_income.head(3)
print("Insight: Top 3 education levels with most high earners:")
for edu, pct in top_edu.items():
    print(f"         {edu}: {pct:.1f}%")


Saved: eda_03_education_analysis.png
Insight: Top 3 education levels with most high earners:
         Doctorate: 74.1%
         Prof-school: 73.4%
         Masters: 55.7%


In [19]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Occupation Analysis', fontsize=14, fontweight='bold')
 


Text(0.5, 0.98, 'Occupation Analysis')

In [20]:
# count per occupation
occ_counts = df['occupation'].value_counts()
axes[0].barh(occ_counts.index, occ_counts.values,
             color='mediumpurple', edgecolor='black', alpha=0.85)
axes[0].set_title('Number of People per Occupation')
axes[0].set_xlabel('Count')

Text(0.5, 0, 'Count')

In [21]:
occ_income = df.groupby('occupation')['is_high_earner'].mean().sort_values() * 100
colors_occ = ['#e74c3c' if x > 40 else '#3498db' for x in occ_income.values]
axes[1].barh(occ_income.index, occ_income.values,
             color=colors_occ, edgecolor='black', alpha=0.85)
axes[1].set_title('% High Earners by Occupation\n(Red = above 40%)')
axes[1].set_xlabel('% Earning >50K')
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0f}%'))
 
plt.tight_layout()
plt.savefig('eda_04_occupation_analysis.png', dpi=150, bbox_inches='tight')
plt.close()
print("\nSaved: eda_04_occupation_analysis.png")
print("Insight: Top occupations for high earners:")
print(occ_income.sort_values(ascending=False).head(3).to_string())


Saved: eda_04_occupation_analysis.png
Insight: Top occupations for high earners:
occupation
Exec-managerial    48.413284
Prof-specialty     34.310114
Protective-serv    32.511556


In [22]:

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Gender Analysis', fontsize=14, fontweight='bold')

Text(0.5, 0.98, 'Gender Analysis')

In [23]:
sex_counts = df['sex'].value_counts()
axes[0].bar(sex_counts.index, sex_counts.values,
            color=['#3498db', '#e91e8c'], edgecolor='black', width=0.4)
axes[0].set_title('Gender Count')
axes[0].set_ylabel('Count')
for i, v in enumerate(sex_counts.values):
    axes[0].text(i, v + 100, f'{v:,}', ha='center', fontweight='bold')

In [24]:
sex_income = df.groupby('sex')['is_high_earner'].mean() * 100
axes[1].bar(sex_income.index, sex_income.values,
            color=['#3498db', '#e91e8c'], edgecolor='black', width=0.4)
axes[1].set_title('% High Earners by Gender')
axes[1].set_ylabel('% Earning >50K')
for i, v in enumerate(sex_income.values):
    axes[1].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontweight='bold')

In [25]:
gender_income = pd.crosstab(df['sex'], df['income'], normalize='index') * 100
gender_income.plot(kind='bar', ax=axes[2],
                   color=['#2ecc71', '#e74c3c'],
                   edgecolor='black', alpha=0.85)
axes[2].set_title('Income Split Within Each Gender (%)')
axes[2].set_xlabel('Gender')
axes[2].set_ylabel('Percentage')
axes[2].tick_params(axis='x', rotation=0)
axes[2].legend(title='Income')
 
plt.tight_layout()
plt.savefig('eda_05_gender_analysis.png', dpi=150, bbox_inches='tight')
plt.close()
print("\nSaved: eda_05_gender_analysis.png")
print(f"Insight: Male high earners  : {sex_income['Male']:.1f}%")
print(f"         Female high earners: {sex_income['Female']:.1f}%")
print("         Significant gender gap in high earning percentage")


Saved: eda_05_gender_analysis.png
Insight: Male high earners  : 30.6%
         Female high earners: 11.0%
         Significant gender gap in high earning percentage


In [26]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Working Hours Analysis', fontsize=14, fontweight='bold')

Text(0.5, 0.98, 'Working Hours Analysis')

In [27]:
axes[0].hist(df['hours_per_week'], bins=25,
             color='darkorange', edgecolor='black', alpha=0.8)
axes[0].axvline(40, color='red', linestyle='--', linewidth=2, label='40hrs standard')
axes[0].axvline(df['hours_per_week'].mean(), color='blue', linestyle='--',
                linewidth=2, label=f"Mean={df['hours_per_week'].mean():.1f}")
axes[0].set_title('Hours Per Week Distribution')
axes[0].set_xlabel('Hours Per Week')
axes[0].set_ylabel('Count')
axes[0].legend()

In [28]:
hrs_income = df.groupby('income')['hours_per_week'].mean()
axes[1].bar(hrs_income.index, hrs_income.values,
            color=['#2ecc71', '#e74c3c'], edgecolor='black', width=0.4)
axes[1].set_title('Avg Hours Per Week by Income Group')
axes[1].set_xlabel('Income')
axes[1].set_ylabel('Avg Hours/Week')
for i, v in enumerate(hrs_income.values):
    axes[1].text(i, v + 0.2, f'{v:.1f}', ha='center', fontweight='bold')
 
plt.tight_layout()
plt.savefig('eda_06_hours_analysis.png', dpi=150, bbox_inches='tight')
plt.close()
print("\nSaved: eda_06_hours_analysis.png")
print(f"Insight: Avg hours for <=50K earners: {hrs_income['<=50K']:.1f} hrs/week")
print(f"         Avg hours for >50K earners : {hrs_income['>50K']:.1f} hrs/week")
print("         High earners work more hours on average")


Saved: eda_06_hours_analysis.png
Insight: Avg hours for <=50K earners: 40.1 hrs/week
         Avg hours for >50K earners : 44.1 hrs/week
         High earners work more hours on average


In [29]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Age Group vs Income', fontsize=14, fontweight='bold')

Text(0.5, 0.98, 'Age Group vs Income')

In [30]:
age_order = ['Youth', 'Young Adult', 'Middle Age', 'Senior']
 
age_income_pct = pd.crosstab(df['age_group'], df['income'],
                              normalize='index')[['<=50K', '>50K']] * 100
age_income_pct = age_income_pct.reindex(age_order)
 
age_income_pct.plot(kind='bar', ax=axes[0],
                    color=['#27ae60', '#c0392b'],
                    edgecolor='black', alpha=0.85)
axes[0].set_title('Income % by Age Group')
axes[0].set_xlabel('Age Group')
axes[0].set_ylabel('Percentage')
axes[0].tick_params(axis='x', rotation=15)
axes[0].legend(title='Income')

In [31]:
age_high = df.groupby('age_group')['is_high_earner'].mean().reindex(age_order) * 100
axes[1].bar(age_order, age_high.values,
            color=['#f39c12', '#3498db', '#9b59b6', '#e74c3c'],
            edgecolor='black', alpha=0.85)
axes[1].set_title('% High Earners per Age Group')
axes[1].set_xlabel('Age Group')
axes[1].set_ylabel('% Earning >50K')
for i, v in enumerate(age_high.values):
    axes[1].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontweight='bold')
 
plt.tight_layout()
plt.savefig('eda_07_age_group_income.png', dpi=150, bbox_inches='tight')
plt.close()
print("\nSaved: eda_07_age_group_income.png")
print("Insight: High earner % by age group:")
for age, pct in age_high.items():
    print(f"         {age}: {pct:.1f}%")


Saved: eda_07_age_group_income.png
Insight: High earner % by age group:
         Youth: 1.8%
         Young Adult: 18.7%
         Middle Age: 36.1%
         Senior: 32.5%


In [32]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Work Class Analysis', fontsize=14, fontweight='bold')
 
wc_counts = df['workclass'].value_counts()
axes[0].barh(wc_counts.index, wc_counts.values,
             color='teal', edgecolor='black', alpha=0.85)
axes[0].set_title('Count per Work Class')
axes[0].set_xlabel('Count')
 
wc_income = df.groupby('workclass')['is_high_earner'].mean().sort_values() * 100
colors_wc = ['#e74c3c' if x > 30 else '#3498db' for x in wc_income.values]
axes[1].barh(wc_income.index, wc_income.values,
             color=colors_wc, edgecolor='black', alpha=0.85)
axes[1].set_title('% High Earners by Work Class')
axes[1].set_xlabel('% Earning >50K')
 
plt.tight_layout()
plt.savefig('eda_08_workclass_analysis.png', dpi=150, bbox_inches='tight')
plt.close()
print("\nSaved: eda_08_workclass_analysis.png")
print("Insight: Top work classes for high earners:")
print(wc_income.sort_values(ascending=False).head(3).to_string())



Saved: eda_08_workclass_analysis.png
Insight: Top work classes for high earners:
workclass
Self-emp-inc    55.734767
Federal-gov     38.645833
Local-gov       29.479216


In [33]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Marital Status Analysis', fontsize=14, fontweight='bold')
 
ms_counts = df['marital_status'].value_counts()
axes[0].bar(range(len(ms_counts)), ms_counts.values,
            color='coral', edgecolor='black', alpha=0.85)
axes[0].set_xticks(range(len(ms_counts)))
axes[0].set_xticklabels(ms_counts.index, rotation=30, ha='right')
axes[0].set_title('Count per Marital Status')
axes[0].set_ylabel('Count')
 
ms_income = df.groupby('marital_status')['is_high_earner'].mean().sort_values() * 100
axes[1].bar(range(len(ms_income)), ms_income.values,
            color='steelblue', edgecolor='black', alpha=0.85)
axes[1].set_xticks(range(len(ms_income)))
axes[1].set_xticklabels(ms_income.index, rotation=30, ha='right')
axes[1].set_title('% High Earners by Marital Status')
axes[1].set_ylabel('% Earning >50K')
for i, v in enumerate(ms_income.values):
    axes[1].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontsize=9, fontweight='bold')
 
plt.tight_layout()
plt.savefig('eda_09_marital_analysis.png', dpi=150, bbox_inches='tight')
plt.close()
print("\nSaved: eda_09_marital_analysis.png")


Saved: eda_09_marital_analysis.png


In [34]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Race Analysis', fontsize=14, fontweight='bold')
 
race_counts = df['race'].value_counts()
axes[0].bar(race_counts.index, race_counts.values,
            color='slateblue', edgecolor='black', alpha=0.85)
axes[0].set_title('Count per Race')
axes[0].set_xlabel('Race')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=20)
 
race_income = df.groupby('race')['is_high_earner'].mean().sort_values() * 100
axes[1].bar(race_income.index, race_income.values,
            color='mediumseagreen', edgecolor='black', alpha=0.85)
axes[1].set_title('% High Earners by Race')
axes[1].set_xlabel('Race')
axes[1].set_ylabel('% Earning >50K')
axes[1].tick_params(axis='x', rotation=20)
for i, v in enumerate(race_income.values):
    axes[1].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontweight='bold')
 
plt.tight_layout()
plt.savefig('eda_10_race_analysis.png', dpi=150, bbox_inches='tight')
plt.close()
print("\nSaved: eda_10_race_analysis.png")


Saved: eda_10_race_analysis.png


In [35]:
num_df = df.select_dtypes(include=np.number)
corr = num_df.corr()
 
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, linecolor='white',
            square=True, ax=ax)
ax.set_title('Correlation Heatmap — All Numeric Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_11_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()
print("\nSaved: eda_11_correlation_heatmap.png")
print("Top features correlated with income:")
income_corr = corr['is_high_earner'].drop('is_high_earner').sort_values(key=abs, ascending=False)
print(income_corr.to_string())


Saved: eda_11_correlation_heatmap.png
Top features correlated with income:
education_num     0.335369
hours_per_week    0.271656
age               0.235586
capital_gain      0.223315
capital_net       0.214417
capital_loss      0.150454
fnlwgt           -0.009517


In [36]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Capital Gain vs Income', fontsize=14, fontweight='bold')

Text(0.5, 0.98, 'Capital Gain vs Income')

In [38]:
cap_nonzero = df[df['capital_gain'] > 0]

In [39]:
cap_nonzero.boxplot(column='capital_gain', by='income', ax=axes[0],
                    patch_artist=True)
axes[0].set_title('Capital Gain Distribution by Income Group')
axes[0].set_xlabel('Income')
axes[0].set_ylabel('Capital Gain')
plt.suptitle('')

Text(0.5, 0.98, '')

In [40]:
avg_cap = cap_nonzero.groupby('income')['capital_gain'].mean()
axes[1].bar(avg_cap.index, avg_cap.values,
            color=['#2ecc71', '#e74c3c'], edgecolor='black', width=0.4)
axes[1].set_title('Avg Capital Gain by Income (non-zero only)')
axes[1].set_xlabel('Income')
axes[1].set_ylabel('Avg Capital Gain ($)')
for i, v in enumerate(avg_cap.values):
    axes[1].text(i, v + 100, f'${v:,.0f}', ha='center', fontweight='bold')
 
plt.tight_layout()
plt.savefig('eda_12_capital_gain.png', dpi=150, bbox_inches='tight')
plt.close()
print("\nSaved: eda_12_capital_gain.png")


Saved: eda_12_capital_gain.png


In [41]:
key_cols = ['age', 'education_num', 'hours_per_week',
            'capital_gain', 'is_high_earner']
sample = df[key_cols + ['income']].sample(1000, random_state=42)
 
fig = sns.pairplot(sample, hue='income',
                   palette={'<=50K': '#2ecc71', '>50K': '#e74c3c'},
                   plot_kws={'alpha': 0.5},
                   diag_kind='hist')
fig.fig.suptitle('Pairplot — Key Numeric Features by Income', y=1.02, fontsize=13)
plt.savefig('eda_13_pairplot.png', dpi=120, bbox_inches='tight')
plt.close()
print("\nSaved: eda_13_pairplot.png")


Saved: eda_13_pairplot.png


In [42]:
print("\n")
print("=" * 60)
print("       EDA COMPLETE — KEY INSIGHTS SUMMARY")
print("=" * 60)
 
total = len(df)
high = df['is_high_earner'].sum()
low  = total - high
 
print(f"""
DATASET
  Total people     : {total:,}
  High earners     : {high:,} ({high/total*100:.1f}%)
  Low earners      : {low:,}  ({low/total*100:.1f}%)
 
AGE
  Overall avg age          : {df['age'].mean():.1f} years
  Avg age of <=50K earners : {df[df['income']=='<=50K']['age'].mean():.1f} years
  Avg age of >50K earners  : {df[df['income']=='>50K']['age'].mean():.1f} years
 
EDUCATION
  Highest earning education : {edu_income.idxmax()}  ({edu_income.max():.1f}%)
  Lowest earning education  : {edu_income.idxmin()}  ({edu_income.min():.1f}%)
 
GENDER
  Male high earners   : {sex_income['Male']:.1f}%
  Female high earners : {sex_income['Female']:.1f}%
 
WORK
  Most common work class : {df['workclass'].mode()[0]}
  Best work class        : {wc_income.idxmax()} ({wc_income.max():.1f}%)
 
HOURS
  Avg hours <=50K : {hrs_income['<=50K']:.1f} hrs/week
  Avg hours >50K  : {hrs_income['>50K']:.1f} hrs/week
 
CORRELATION WITH INCOME
  Strongest feature : {income_corr.index[0]} ({income_corr.iloc[0]:+.3f})
  2nd strongest     : {income_corr.index[1]} ({income_corr.iloc[1]:+.3f})
  3rd strongest     : {income_corr.index[2]} ({income_corr.iloc[2]:+.3f})
""")
 
print("Charts generated:")
charts = [
    "eda_01_income_distribution.png",
    "eda_02_age_analysis.png",
    "eda_03_education_analysis.png",
    "eda_04_occupation_analysis.png",
    "eda_05_gender_analysis.png",
    "eda_06_hours_analysis.png",
    "eda_07_age_group_income.png",
    "eda_08_workclass_analysis.png",
    "eda_09_marital_analysis.png",
    "eda_10_race_analysis.png",
    "eda_11_correlation_heatmap.png",
    "eda_12_capital_gain.png",
    "eda_13_pairplot.png"
]
for c in charts:
    print(f"  📊 {c}")
 
print("\nDone! Task 02 EDA complete.")



       EDA COMPLETE — KEY INSIGHTS SUMMARY

DATASET
  Total people     : 32,521
  High earners     : 7,839 (24.1%)
  Low earners      : 24,682  (75.9%)

AGE
  Overall avg age          : 38.6 years
  Avg age of <=50K earners : 36.8 years
  Avg age of >50K earners  : 44.2 years

EDUCATION
  Highest earning education : Doctorate  (74.1%)
  Lowest earning education  : Preschool  (0.0%)

GENDER
  Male high earners   : 30.6%
  Female high earners : 11.0%

WORK
  Most common work class : Private
  Best work class        : Self-emp-inc (55.7%)

HOURS
  Avg hours <=50K : 40.1 hrs/week
  Avg hours >50K  : 44.1 hrs/week

CORRELATION WITH INCOME
  Strongest feature : education_num (+0.335)
  2nd strongest     : hours_per_week (+0.272)
  3rd strongest     : age (+0.236)

Charts generated:
  📊 eda_01_income_distribution.png
  📊 eda_02_age_analysis.png
  📊 eda_03_education_analysis.png
  📊 eda_04_occupation_analysis.png
  📊 eda_05_gender_analysis.png
  📊 eda_06_hours_analysis.png
  📊 eda_07_age_gro